In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

REPORT = Path("/workspace/results/analysis/tls_grid_a3_01")

tables = {
    name: pd.read_csv(
        REPORT / f"{name}.csv",
        dtype=object,
        keep_default_na=False,
        na_values=[r"\N"],
    )
    for name in (
        "resource_builds",
        "resource_observations",
        "resource_distributions",
        "resource_differences",
    )
}

In [2]:
summary = tables["resource_distributions"]

metrics = [
    "flash_section_bytes",
    "application_binary_bytes",
    "ram_static_base_bytes",
    "ram3_section_bytes",
    "freertos_heap_peak_used_bytes",
    "sbrk_peak_used_bytes",
    "msp_peak_used_bytes",
    "mldsa_workspace_peak_used_bytes",
]

display(summary.loc[
    summary["metric"].isin(metrics),
    [
        "condition", "metric", "median_bytes", "median_KiB",
        "sample_unit", "n_observations",
        "unavailable_or_excluded", "reason",
    ],
])

,condition,metric,median_bytes,median_KiB,sample_unit,n_observations,unavailable_or_excluded,reason
0,ML_KEM_768__mldsa65__mutual,flash_section_bytes,283576,276.9296875,unique_build,1,0,NaN
2,ML_KEM_768__mldsa65__mutual,application_binary_bytes,283576,276.9296875,unique_build,1,0,NaN
3,ML_KEM_768__mldsa65__mutual,ram_static_base_bytes,154880,151.25,unique_build,1,0,NaN
6,ML_KEM_768__mldsa65__mutual,ram3_section_bytes,131072,128.0,unique_build,1,0,NaN
10,ML_KEM_768__mldsa65__mutual,freertos_heap_peak_used_bytes,83456.0,81.5,run_lifetime,10,0,NaN
...,...,...,...,...,...,...,...,...
259,secp256r1__rsa3072__server,ram3_section_bytes,32768,32.0,unique_build,1,0,NaN
263,secp256r1__rsa3072__server,freertos_heap_peak_used_bytes,63244.0,61.76171875,run_lifetime,10,0,NaN
268,secp256r1__rsa3072__server,sbrk_peak_used_bytes,32.0,0.03125,run_lifetime,10,0,NaN
271,secp256r1__rsa3072__server,msp_peak_used_bytes,1048.0,1.0234375,run_lifetime,10,0,NaN


In [4]:
import json
from statistics import fmean
import pandas as pd
from IPython.display import display

# Reuse A3's eligible observation identities, including its exclusions.
records = []

for row in tables["resource_distributions"].itertuples(index=False):
    ids = (
        json.loads(row.observation_ids)
        if isinstance(row.observation_ids, str)
        else row.observation_ids
    )

    if row.sample_unit == "unique_build":
        source, identity = tables["resource_builds"], "build_id"
    else:
        source, identity = tables["resource_observations"], "run_key"

    selected = source.loc[
        source["metric"].eq(row.metric)
        & source[identity].isin(ids)
        & source["validity"].eq("valid")
    ]

    # Guard against accidentally counting a build/run more than once.
    assert not selected[identity].duplicated().any()
    values = [int(value) for value in selected["value_bytes"]]
    assert len(values) == int(row.n_observations)

    records.append({
        "condition": row.condition,
        "metric": row.metric,
        "mean_bytes": fmean(values) if values else None,
        "n": len(values),
    })

averages = pd.DataFrame(records)

# Logical profile order for this TLS grid.
kex_order = ["secp256r1", "SecP256r1MLKEM768", "ML_KEM_768"]
cert_order = ["rsa3072", "mldsa65"]
auth_order = ["server", "mutual"]

def profile_order(condition):
    kex, cert, auth = condition.split("__")
    return (
        kex_order.index(kex),
        cert_order.index(cert),
        auth_order.index(auth),
    )

profiles = sorted(averages["condition"].unique(), key=profile_order)

# A3 already orders measurements by static footprint, heap, newlib,
# MSP, workspace, then the explicitly unavailable combined peak.
measurements = list(
    tables["resource_distributions"]["metric"].drop_duplicates()
)

mean_bytes = averages.pivot(
    index="metric", columns="condition", values="mean_bytes"
).reindex(index=measurements, columns=profiles)

sample_counts = averages.pivot(
    index="metric", columns="condition", values="n"
).reindex(index=measurements, columns=profiles)

# Three header levels make the twelve profiles easier to scan.
headers = pd.MultiIndex.from_tuples(
    [tuple(profile.split("__")) for profile in profiles],
    names=["Key exchange", "Certificate", "Authentication"],
)

mean_bytes.columns = headers
sample_counts.columns = headers

labels = [
    metric.removesuffix("_bytes").replace("_", " ")
    for metric in measurements
]
mean_bytes.index = pd.Index(labels, name="Measurement")
sample_counts.index = mean_bytes.index

mean_kib = mean_bytes / 1024

display(
    mean_kib.style
    .format("{:,.3f}", na_rep="—")
    .set_caption("Mean application resources — KiB (1 KiB = 1024 bytes)")
    .set_sticky(axis="index")
    .set_sticky(axis="columns")
)